In [1]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ Successfully installed {package}")
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed to install {package}: {e}")

# List of required packages
required_packages = [
    "numpy",
    "scipy", 
    "cupy-cuda12x",  # Adjust CUDA version as needed (cupy-cuda11x for CUDA 11.x)
    "psutil",
    "pynvml",
    "pandas",
    "openpyxl",
    "pathlib"
]

print("Installing required packages...")
for package in required_packages:
    install_package(package)

print("\nInstallation complete!")

Installing required packages...
✓ Successfully installed numpy
✓ Successfully installed scipy
✓ Successfully installed cupy-cuda12x
✓ Successfully installed psutil
✓ Successfully installed pynvml
✓ Successfully installed pandas
✓ Successfully installed openpyxl
✓ Successfully installed pathlib

Installation complete!


In [ ]:
# Quick install - uncomment and run to install specific packages
# !pip install numpy
# !pip install scipy
# !pip install cupy-cuda12x  # For CUDA 12.x, use cupy-cuda11x for CUDA 11.x
# !pip install psutil
# !pip install pynvml
# !pip install pandas
# !pip install openpyxl

# Alternative: Install multiple packages at once
# !pip install numpy scipy psutil pynvml pandas openpyxl

# For GPU support (choose the right CUDA version):
# !pip install cupy-cuda11x  # For CUDA 11.x
# !pip install cupy-cuda12x  # For CUDA 12.x

print("Uncomment the lines above to install packages")

In [2]:
import time
import psutil
import os
import threading
import traceback
import random
import copy
from pynvml import *
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

In [22]:
import numpy as np
import cmath    
import math 

class GateType(Enum):
    X = auto()
    Y = auto()
    Z = auto()
    H = auto()
    S = auto()
    SDG = auto()
    T = auto()
    TDG = auto()
    SX = auto()
    V = auto()
    V_plus = auto()
    ARBITRARY = auto()

GATE_UNITARIES: Dict[GateType, List[List[complex]]]= {
    GateType.X: [[0, 1],
                [1, 0]],
    GateType.Y: [[0, -1j],
                [1j, 0]],
    GateType.Z: [[1, 0],
                [0, -1]],
    GateType.H: [[1/2**0.5, 1/2**0.5],
                [1/2**0.5, -1/2**0.5]],
    GateType.S: [[1, 0], 
                [0,  1j]],
    GateType.SDG: [[1, 0], 
                  [0, -1j]],
    GateType.T:   [[1, 0], 
                  [0,  cmath.exp(1j*math.pi/4)]],
    GateType.TDG: [[1, 0], 
                  [0,  cmath.exp(-1j*math.pi/4)]],
    GateType.SX:  [[0.5+0.5j, 0.5-0.5j],
                  [0.5-0.5j, 0.5+0.5j]],
    GateType.V:   [[0.5+0.5j, 0.5-0.5j], 
                  [0.5-0.5j, 0.5+0.5j]],
    GateType.V_plus:  [[0.5-0.5j, 0.5+0.5j], 
                  [0.5+0.5j, 0.5-0.5j]],
    GateType.ARBITRARY: [[1, 0],
                        [0, 1]],  # Default to identity, can be overridden
}

@dataclass
class SingleQubitGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int

@dataclass
class ControlledGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int
    control: List[int]

GateOperation = SingleQubitGate | ControlledGate

def operation_constructor(gate: GateType, 
                          target: int, 
                          control: Optional[List[int]] = None, 
                          mode: Optional[str] = None,
                          matrix: Optional[List[List[complex]]] = None,
                          angles: Optional[List[float]] = None
) -> GateOperation:
    
    """
    Construct a quantum gate operation, either from a built-in gate type or an arbitrary unitary.

    This function returns either a SingleQubitGate or a ControlledGate depending on whether
    control qubits are specified. For GateType.ARBITRARY, you must specify `mode` as either
    'angles' (to generate the U3 matrix from Euler angles) or 'matrix' (to supply your own
    2x2 unitary).

    Args:
        gate (GateType): 
            The gate type to construct. Use GateType.ARBITRARY for custom unitaries.
        target (int): 
            Index of the target qubit.
        control (Optional[List[int]]): 
            List of control-qubit indices. If None, a single-qubit gate is returned.
        mode (Optional[str]): 
            Only for GateType.ARBITRARY.  
            - 'angles': compute the unitary from `angles`.  
            - 'matrix': validate and use the provided `matrix`.
        matrix (Optional[List[List[complex]]]): 
            A 2x2 complex matrix for ARBITRARY gates when `mode='matrix'`.
        angles (Optional[List[float]]): 
            Three Euler angles [theta, phi, lambda] for ARBITRARY gates when `mode='angles'`.

    Returns:
        GateOperation: 
            A SingleQubitGate or ControlledGate instance encapsulating the specified operation.

    Raises:
        ValueError: 
            - If `mode` is missing or invalid for an ARBITRARY gate.  
            - If `angles` is not a list of exactly three floats when `mode='angles'`.  
            - If `matrix` is missing, not 2x2, or not unitary when `mode='matrix'`.
    """

    def is_unitary(matrix: List[List[complex]], tol=1e-8) -> bool:
        M = np.array(matrix, dtype=complex)
        return np.allclose(M.conj().T @ M, np.eye(2), atol=tol)
    
    def check_matrix(matrix: List[List[complex]]):
        if matrix is None:
            raise ValueError("ARBITRARY gate requires a 2x2 matrix.")
        if len(matrix) != 2 or len(matrix[0]) != 2 or len(matrix[1]) != 2:
            raise ValueError("Matrix must be 2x2 for arbitrary gate.")
        if not is_unitary(matrix):
            raise ValueError("Provided matrix is not unitary.")
    
    def handle_angles(angles: List[float]) -> List[List[complex]]:
        if angles is None or len(angles) != 3:
            raise ValueError("Angles must be provided as a list of three values (theta, phi, lambda).")
        theta, phi, lam = angles
        a = np.cos(theta / 2)
        b = -np.exp(1j * lam) * np.sin(theta / 2)
        c = np.exp(1j * phi) * np.sin(theta / 2)
        d = np.exp(1j * (phi + lam)) * np.cos(theta / 2)
        matrix = [[a, b], [c, d]]
        return matrix

    if gate is GateType.ARBITRARY:
        if mode is None:
            raise ValueError("Mode must be specified for ARBITRARY gate.")
        if mode == 'angles':
            matrix = handle_angles(angles)
        elif mode == 'matrix':
            check_matrix(matrix)
        else:
            raise ValueError("Invalid mode for ARBITRARY gate. Use 'angles' or 'matrix'.")
    else:
        matrix = GATE_UNITARIES[gate]
    
    a, b = matrix[0]
    c, d = matrix[1]
    
    if control is None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)


'''
def customize(gate: GateType, target: int, control: List[int] | None = None) -> GateOperation:
    matrix = GATE_UNITARIES[gate]
    a, b = matrix[0]
    c, d = matrix[1]
    if control == None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)

def customize_arbitrary(matrix: List[List[complex]], target: int, control: List[int] | None = None) -> GateOperation:
    """
    Create a gate operation with an arbitrary 2x2 unitary matrix.
    
    Args:
        matrix: 2x2 unitary matrix as [[a, b], [c, d]]
        target: Target qubit index
        control: Optional list of control qubits
    
    Returns:
        GateOperation with the custom matrix
    
    Example:
        # Create a rotation gate around X-axis by π/4
        import math
        cos_val = math.cos(math.pi/8)
        sin_val = math.sin(math.pi/8)
        rx_matrix = [[cos_val, -1j*sin_val], [-1j*sin_val, cos_val]]
        gate = customize_arbitrary(rx_matrix, target=0)
    """
    if len(matrix) != 2 or len(matrix[0]) != 2 or len(matrix[1]) != 2:
        raise ValueError("Matrix must be 2x2")
    
    a, b = matrix[0]
    c, d = matrix[1]
    
    if control == None:
        return SingleQubitGate(GateType.ARBITRARY, a, b, c, d, target)
    else:
        return ControlledGate(GateType.ARBITRARY, a, b, c, d, target, control)
'''

def randomizer(option: Optional[GateOperation], threshold=0) -> List[GateOperation]:
    operations: List[GateOperation] = []

    for _ in range(NUM_OPERATIONS):
        gate = random.choice(list(GateType))
        matrix = GATE_UNITARIES[gate]
        a, b = matrix[0]
        c, d = matrix[1]
        
        if option == SingleQubitGate or random.random() < threshold:
            target = random.randint(0, NUM_QUBITS - 1)
            operations.append(SingleQubitGate(gate, a, b, c, d, target))
        elif option == ControlledGate or random.random() >= threshold:
            control, target = random.sample(range(NUM_QUBITS), 2)
            operations.append(ControlledGate(gate, a, b, c, d, target, [control]))
    
    return operations

def initialization(L, R) -> Tuple[List[List[complex]], List[List[complex]], List[List[int]]]:    
    if 1 < INIT_MATRIX_SIZE:
        L[1, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            L[row, col] = 1

    if 2 < INIT_MATRIX_SIZE:
        R[2, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        if col % 2 == 1:  
            row = col + 3
        else:  
            row = col + 2
        if row < INIT_MATRIX_SIZE:
            R[row, col] = 1

    Mapping = []
    Mapping.append([0])

    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*i + 1])
    Mapping.append([2*i + 2])

    return L, R, Mapping

def initialization_sparse(INIT_MATRIX_SIZE, NUM_QUBITS, TYPE):
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
        
    row_L, col_L, data_L = [], [], []
    row_R, col_R, data_R = [], [], []

    if 1 < INIT_MATRIX_SIZE:
        row_L.append(1)
        col_L.append(0)
        data_L.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            row_L.append(row)
            col_L.append(col)
            data_L.append(1)

    if 2 < INIT_MATRIX_SIZE:
        row_R.append(2)
        col_R.append(0)
        data_R.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 3 if col % 2 == 1 else col + 2
        if row < INIT_MATRIX_SIZE:
            row_R.append(row)
            col_R.append(col)
            data_R.append(1)

    L = sparse.csr_matrix((cp.array(data_L, dtype=cp.complex128), (cp.array(row_L), cp.array(col_L))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    R = sparse.csr_matrix((cp.array(data_R, dtype=cp.complex128), (cp.array(row_R), cp.array(col_R))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    Mapping = [[0]]
    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*NUM_QUBITS - 1])
    Mapping.append([2*NUM_QUBITS])

    return L, R, Mapping
def _view_operations(operations: List[GateOperation]):
    for op in operations:
        if op.gate == GateType.ARBITRARY:
            print(f"ARBITRARY({op.a}, {op.b}, {op.c}, {op.d})", op.target)
        else:
            print(op.gate.name, op.target)

In [23]:
### Fixed QASM parser with proper imports
import math, cmath, re
from pathlib import Path
from typing import Tuple, List

_PI_ENV = {"pi": math.pi, "tau": 2*math.pi, "e": math.e}

def _eval_expr(expr: str) -> float:
    """安全地把 'pi/2', '3*pi/4', '1.570796'… 轉成 float"""
    return eval(expr, {"__builtins__": None}, _PI_ENV)

def _rot_x(theta: float):
    c, s = math.cos(theta/2), math.sin(theta/2)
    return [[c, -1j*s], [-1j*s, c]]

def _rot_y(theta: float):
    c, s = math.cos(theta/2), math.sin(theta/2)
    return [[c, -s], [s, c]]

def _rot_z(theta: float):
    return [[cmath.exp(-1j*theta/2), 0],
            [0, cmath.exp(1j*theta/2)]]

def _phase(lambda_):
    return [[1, 0], [0, cmath.exp(1j*lambda_)]]

def extract_qubit_index(s: str) -> int:
    """Extract qubit index from string like 'q[0]' -> 0"""
    match = re.search(r'\[(\d+)\]', s)
    if match:
        return int(match.group(1))
    raise ValueError(f"Invalid qubit format: {s}")
## .reals file parser for revlib
def parse_reals(path: Path) -> Tuple[int, List[GateOperation]]:
    gate_str_to_enum = { 'v' : GateType.V, 'v+': GateType.V_plus}
    num_qubits, gate_list = None, []
    with open(path, 'r') as f:
        for raw in f:
            line = raw.strip().rstrip('\n')
            if not line or line.startswith('#'):
                continue
            # version ,varialbe, inputs, outputs, constants, garbage line (ignore)
            if line.startswith('.version') or line.startswith('.variable') or line.startswith('.inputs') or line.startswith('.outputs') or line.startswith('.constants') or line.startswith('.garbage') or line.startswith('.begin') or line.startswith('.end'):
                continue
            # numvars line
            if line.startswith('.numvars'):
                m = re.search(r'\d+', line)
                if m:
                    num_qubits = int(m.group(0))
                continue
            # gate line
            m = re.match(r'([a-zA-Z_][\w]*)(\([^)]*\))?\s+(.*)', line.strip())
            if not m:
                print(f"Malformed line: {line}"); continue
            token, para, qubits_part = m.groups()
            qubit_strs = [q.strip() for q in qubits_part.split()]
            ## deal with v v+ gate
            if token in {'v', 'v+'}:
                if len(qubit_strs) != 2:
                    print(f"Control/target mismatch in: {line}"); continue
                target = extract_qubit_index(qubit_strs[1])
                ctrls = [qubit_strs[0]]
                gate_enum = gate_str_to_enum[token]
                gate_op = operation_constructor(gate_enum, target, ctrls)
                gate_list.append(gate_op)
                continue
            if token.startswith('t'):
                if len(qubit_strs) == 1:
                    target = extract_qubit_index(qubit_strs[-1])
                    gate_op = operation_constructor(GateType.X, target)
                else:
                    target = extract_qubit_index(qubit_strs[-1])
                    ctrls = [extract_qubit_index(q) for q in qubit_strs[:-1]]
                    gate_op = operation_constructor(GateType.X, target, ctrls)
                gate_list.append(gate_op)
                continue
            if token.startswith('f'): 
                if len(qubit_strs) == 1:
                    print(f"Control/target mismatch in: {line}"); continue
                else:
                    target1 = extract_qubit_index(qubit_strs[-1])
                    target2 = extract_qubit_index(qubit_strs[-2])
                    ctrls = [extract_qubit_index(q) for q in qubit_strs]
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target2, list(set(ctrls) - set([target2])),
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                continue
            if token.startswith('p'):
                if len(qubit_strs) != 3:
                    print(f"Control/target mismatch in: {line}"); continue
                else:
                    target1 = extract_qubit_index(qubit_strs[-2])
                    target2 = extract_qubit_index(qubit_strs[-1])
                    ctrls1 = [extract_qubit_index(q) for q in qubit_strs[:-2]]
                    ctrls2 = [extract_qubit_index(q) for q in qubit_strs[:-1]]
                    if token == 'pi' :
                        gate_list.append(
                            operation_constructor(GateType.X, target1, ctrls1)
                        )
                        gate_list.append(
                            operation_constructor(GateType.X, target2, ctrls2)
                        )
                        continue
                    else :
                        gate_list.append(
                            operation_constructor(GateType.X, target2, ctrls2)
                        )
                        gate_list.append(
                            operation_constructor(GateType.X, target1, ctrls1)
                        )
                        continue
    if num_qubits is None:
        raise ValueError("No .numvars line found in the file.")
    return num_qubits, gate_list
        

def parse_qasm(path: Path) -> Tuple[int, List[GateOperation]]:
    # 固定閘對應表
    gate_str_to_enum = {
        'x': GateType.X,  'y': GateType.Y,  'z': GateType.Z,  'h': GateType.H,
        's': GateType.S,  'sdg': GateType.SDG,
        't': GateType.T,  'tdg': GateType.TDG,
        'sx': GateType.SX,
    }

    num_qubits, gate_list = 0, []

    with open(path, 'r') as f:
        for raw in f:
            line = raw.strip().rstrip(';')
            if (not line or line.startswith('//')
                    or line.startswith('OPENQASM')
                    or line.startswith('include')):
                continue

            # qreg 宣告
            if line.startswith('qreg'):
                m = re.search(r'qreg\s+\w+\[(\d+)\]', line)
                if m:
                    num_qubits = num_qubits + int(m.group(1))
                continue
            # creg 宣告（忽略）
            if line.startswith('creg'):
                continue

            # barrier 宣告（忽略）
            if line.startswith('barrier'):
                continue
            # measure 宣告（忽略）
            if line.startswith('measure'):
                continue
            # ===== 解析 gate token、參數、qubit 清單 =====
            m = re.match(r'([a-zA-Z_][\w]*)(\([^)]*\))?\s+(.*)', line)
            if not m:
                print(f"Malformed line: {line}"); continue
            token, param_group, qubits_part = m.groups()
            qubit_strs = [q.strip() for q in qubits_part.split(',')]

            # ── 拆出控制字首 (c、cc …) ──
            ctrl_prefix = re.match(r'c+', token)
            num_ctrl = len(ctrl_prefix.group()) if ctrl_prefix else 0
            base_token = token[num_ctrl:]

            #處理swap gate (拆成3個cnot)
            if base_token == 'swap' or base_token == 'cswap':
                if len(qubit_strs) != 2 + num_ctrl:
                    print(f"Control/target mismatch in: {line}"); continue
                ctrls = [extract_qubit_index(q) for q in qubit_strs] 
                target1 = extract_qubit_index(qubit_strs[-2])
                target2 = extract_qubit_index(qubit_strs[-1])
                print(ctrls, target1, target2)
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target2, list(set(ctrls) - set([target2])),
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                continue

            # ── 取得角度參數（若有） ──
            params = []
            if param_group:                       # "(theta,phi,…)" → list[str]
                params = [p.strip() for p in param_group[1:-1].split(',')]
                params = [_eval_expr(p) for p in params]

            # ── 驗證 qubit 數 ──
            if len(qubit_strs) != (num_ctrl + 1):
                print(f"Control/target mismatch in: {line}"); continue
            ctrls = [extract_qubit_index(q) for q in qubit_strs[:-1]]
            target = extract_qubit_index(qubit_strs[-1])

            # ===== 對應到 GateOperation =====
            #
            # 1. 固定單量子閘
            if not params and base_token in gate_str_to_enum:
                gate_enum = gate_str_to_enum[base_token]
                gate_op = operation_constructor(gate_enum, target,
                                                ctrls if num_ctrl else None)
                gate_list.append(gate_op); continue

            # 2. 參數化單量子閘（rx/ry/rz/p/u1/u2/u3）
            if base_token in {'rx','ry','rz','p','phase','u1','u2','u3'}:
                # 生成對應矩陣
                if base_token == 'rx': M = _rot_x(params[0])
                elif base_token == 'ry': M = _rot_y(params[0])
                elif base_token == 'rz': M = _rot_z(params[0])
                elif base_token in ['p','phase','u1']: M = _phase(params[0])
                elif base_token == 'u2':
                    phi, lam = params
                    # U2 gate: RZ(phi) · RY(π/2) · RZ(lam)
                    c, s = math.cos(math.pi/4), math.sin(math.pi/4)
                    M = [[c, -s*cmath.exp(1j*lam)], 
                         [s*cmath.exp(1j*phi), c*cmath.exp(1j*(phi+lam))]]
                elif base_token == 'u3':
                    theta, phi, lam = params
                    # U3 gate: general single-qubit unitary
                    c, s = math.cos(theta/2), math.sin(theta/2)
                    M = [[c, -s*cmath.exp(1j*lam)], 
                         [s*cmath.exp(1j*phi), c*cmath.exp(1j*(phi+lam))]]
                
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target,
                                          ctrls if num_ctrl else None,
                                          mode='matrix', matrix=M)
                )
                continue

            # 3. 控制版（cp/crx/cry/crz）
            if base_token in {'p','rx','ry','rz'} and num_ctrl >= 1:
                # 重用上面矩陣生成功能
                if base_token == 'p': M = _phase(params[0])
                elif base_token == 'rx': M = _rot_x(params[0])
                elif base_token == 'ry': M = _rot_y(params[0])
                elif base_token == 'rz': M = _rot_z(params[0])
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target,
                                          ctrls, mode='matrix', matrix=M)
                )
                continue

            print(f"Skipping unsupported or malformed line: {line}")

    if num_qubits is None:
        raise ValueError("No qreg declaration found in QASM file.")

    return num_qubits, gate_list

print("✅ QASM parser functions defined successfully!")


def parse_qasm_to_custom(path: Path):
    if path.suffix == '.qasm':
        return parse_qasm(path)
    elif path.suffix == '.real':
        return parse_reals(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

✅ QASM parser functions defined successfully!


In [13]:
### test parser
path = Path("./DD_benchmark/RevLib/reals/ADD/add6_19.real")
print(parse_reals_to_custom(path))  # Replace with your QASM file path

Control/target mismatch in: creg c[3]
Control/target mismatch in: barrier q[0],q[1],q[2],q[3]
Skipping unsupported or malformed line: measure q[0] -> c[0]
Skipping unsupported or malformed line: measure q[1] -> c[1]
Skipping unsupported or malformed line: measure q[2] -> c[2]
(4, [SingleQubitGate(gate=<GateType.ARBITRARY: 10>, a=0.7071067811865476, b=(-0.7071067811865475+0j), c=(0.7071067811865475+0j), d=(0.7071067811865476+0j), target=0), SingleQubitGate(gate=<GateType.ARBITRARY: 10>, a=0.7071067811865476, b=(-0.7071067811865475+0j), c=(0.7071067811865475+0j), d=(0.7071067811865476+0j), target=1), SingleQubitGate(gate=<GateType.H: 4>, a=0.7071067811865475, b=0.7071067811865475, c=0.7071067811865475, d=-0.7071067811865475, target=2), SingleQubitGate(gate=<GateType.ARBITRARY: 10>, a=0.7071067811865476, b=(0.7071067811865475+8.659560562354932e-17j), c=(-0.7071067811865475-8.659560562354932e-17j), d=(0.7071067811865476+1.7319121124709868e-16j), target=3), ControlledGate(gate=<GateType.X: 

In [9]:
NUM_QUBITS = 3
# NUM_OPERATIONS = 20
INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

# operations = randomizer(option=ControlledGate)

operations : List[GateOperation] = []
operations.append(customize(GateType.X, 0))
operations.append(customize(GateType.X, 2))
# operations.append(customize(GateType.H, 0))
# operations.append(customize(GateType.H, 1))
# operations.append(customize(GateType.X, 0, [1]))
# operations.append(customize(GateType.H, 0))
# operations.append(customize(GateType.H, 1))
operations.append(customize(GateType.X, 1, [0, 2]))
# operations.append(customize(GateType.X, 2, [1]))

# operations.append(customize(GateType.Z, 2))

# _view_operations(operations)

In [11]:
# Examples of arbitrary unitary matrices
import math
import cmath

# Example 1: Rotation around X-axis by π/4
cos_val = math.cos(math.pi/8)
sin_val = math.sin(math.pi/8)
rx_pi4_matrix = [[cos_val, -1j*sin_val], 
                 [-1j*sin_val, cos_val]]

# Example 2: Rotation around Y-axis by π/6
cos_val_y = math.cos(math.pi/12)
sin_val_y = math.sin(math.pi/12)
ry_pi6_matrix = [[cos_val_y, -sin_val_y], 
                 [sin_val_y, cos_val_y]]

# Example 3: Rotation around Z-axis by π/3
cos_val_z = math.cos(math.pi/6)
sin_val_z = math.sin(math.pi/6)
rz_pi3_matrix = [[cos_val_z - 1j*sin_val_z, 0], 
                 [0, cos_val_z + 1j*sin_val_z]]

# Example 4: Phase gate (S gate)
s_gate_matrix = [[1, 0], 
                 [0, 1j]]

# Example 5: T gate
t_gate_matrix = [[1, 0], 
                 [0, cmath.exp(1j*math.pi/4)]]

print("Available arbitrary unitary examples:")
print("1. RX(π/4) - Rotation around X-axis")
print("2. RY(π/6) - Rotation around Y-axis") 
print("3. RZ(π/3) - Rotation around Z-axis")
print("4. S gate - Phase gate")
print("5. T gate - π/8 phase gate")

# Create some example operations with arbitrary matrices
example_operations = []
example_operations.append(customize_arbitrary(rx_pi4_matrix, target=0))
example_operations.append(customize_arbitrary(ry_pi6_matrix, target=1))
example_operations.append(customize_arbitrary(s_gate_matrix, target=2))
example_operations.append(customize_arbitrary(t_gate_matrix, target=0, control=[1]))

print("\nExample operations created:")
_view_operations(example_operations)

Available arbitrary unitary examples:
1. RX(π/4) - Rotation around X-axis
2. RY(π/6) - Rotation around Y-axis
3. RZ(π/3) - Rotation around Z-axis
4. S gate - Phase gate
5. T gate - π/8 phase gate

Example operations created:
ARBITRARY(0.9238795325112867, -0.3826834323650898j, -0.3826834323650898j, 0.9238795325112867) 0
ARBITRARY(0.9659258262890683, -0.25881904510252074, 0.25881904510252074, 0.9659258262890683) 1
ARBITRARY(1, 0, 0, 1j) 2
ARBITRARY(1, 0, 0, (0.7071067811865476+0.7071067811865475j)) 0


In [12]:
def compute_result(L, R, seq: List[int], leaf: List[int], TYPE) -> complex:
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
        
    A = cp.zeros((1, L.shape[0]))

    A[0, leaf] = 1

    B = cp.zeros((L.shape[0], 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

class SimulationAllPath:
    def __init__(self, L, R, leaf, NUM_QUBITS, TYPE):
        self.L = L
        self.R = R
        self.LL = self.L@self.L
        self.RR = self.R@self.R
        self.LR = self.L@self.R
        self.RL = self.R@self.L
        self.leaf = leaf
        self.NUM_QUBITS = NUM_QUBITS
        self.TYPE = TYPE
        self.sim_result = {}

    def recursion(self, seq: List[int], dirs, A, start_time):
        if time.time() - start_time > 60:
            raise TimeoutError("Simulation Timeout: 60s.")
        if len(seq) == self.NUM_QUBITS:
            result = A @ dirs
            val = result.toarray()[0, 0]
            if abs(val) > 1e-10:
                self.sim_result[tuple(seq)] = val
            return
        
        if self.NUM_QUBITS <= 10 or len(seq) < 2:
            for dir in [0, 1]:
                next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                self.recursion(seq + [dir], next_dirs, A, start_time)
        else:
            for d1 in [0, 1]:
                for d2 in [0, 1]:
                    if d1 == 0 and d2 == 0:
                        next_dirs = self.LL @ dirs
                    elif d1 == 0 and d2 == 1:
                        next_dirs = self.RL @ dirs
                    elif d1 == 1 and d2 == 0:
                        next_dirs = self.LR @ dirs
                    elif d1 == 1 and d2 == 1:
                        next_dirs = self.RR @ dirs
                    
                    self.recursion(seq + [d1, d2], next_dirs, A, start_time)

    def simulate_non_recursive(self, A, B, start_time):
        stack = [([], B)]  # (sequence_so_far, current_dirs)

        while stack:
            if time.time() - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            if self.NUM_QUBITS <= 10 or len(seq) == self.NUM_QUBITS - 1:
                for dir in [0, 1]:
                    next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                    stack.append((seq + [dir], next_dirs))
            else:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        if d1 == 0 and d2 == 0:
                            next_dirs = self.LL @ dirs
                        elif d1 == 0 and d2 == 1:
                            next_dirs = self.RL @ dirs
                        elif d1 == 1 and d2 == 0:
                            next_dirs = self.LR @ dirs
                        else:  # d1 == 1 and d2 == 1
                            next_dirs = self.RR @ dirs

                        stack.append((seq + [d1, d2], next_dirs))

    def build_4bit_matrix_map(self):
        from itertools import product

        mat_map = {}
        for bits in product([0, 1], repeat=4):
            mat = self.L if bits[0] == 0 else self.R
            for b in bits[1:]:
                mat = (self.L if b == 0 else self.R) @ mat
            mat_map[bits] = mat
        return mat_map
    
    def simulate_non_recursive_4bit(self, A, B, start_time):
        matrix4_map = self.build_4bit_matrix_map()
        stack = [([], B)]

        while stack:
            now_time = time.time()
            if now_time - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            remaining = self.NUM_QUBITS - len(seq)
            if remaining >= 4:
                for bits4, mat in matrix4_map.items():
                    next_seq = seq + list(bits4)
                    next_dirs = mat @ dirs
                    stack.append((next_seq, next_dirs))
            elif remaining >= 2:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        next_dirs = (
                            self.LL @ dirs if d1 == 0 and d2 == 0 else
                            self.RL @ dirs if d1 == 0 and d2 == 1 else
                            self.LR @ dirs if d1 == 1 and d2 == 0 else
                            self.RR @ dirs
                        )
                        stack.append((seq + [d1, d2], next_dirs))
            else:
                for d in [0, 1]:
                    next_dirs = self.L @ dirs if d == 0 else self.R @ dirs
                    stack.append((seq + [d], next_dirs))
            
            print(time.time() - now_time, "seconds elapsed for current iteration:", seq)


    def simulation(self):
        if self.TYPE == 'numpy':
            import numpy as cp
            from scipy import sparse
        elif self.TYPE == 'cupy':
            import cupy as cp
            from cupyx.scipy import sparse

        A = cp.zeros((1, self.L.shape[0]))
        A[0, self.leaf] = 1
        A = sparse.csr_matrix(A, dtype=cp.complex128)

        B = cp.zeros((self.L.shape[0], 1))
        B[0, 0] = 1
        B = sparse.csr_matrix(B, dtype=cp.complex128)

        start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)
        self.simulate_non_recursive_4bit(A, B, start_simulation)


    def __iter__(self):
        ...
        # A = cp.zeros((1, self.L.shape[0]))
        # A[0, self.leaf] = 1
        # A = sparse.csc_matrix(A, dtype=cp.complex128)

        # B = cp.zeros((self.L.shape[0], 1))
        # B[0, 0] = 1
        # B = sparse.csc_matrix(B, dtype=cp.complex128)

        # start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)

        # for i in range(2**self.NUM_QUBITS):
        #     if time.time() - start_simulation > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")

        #     seq = [(i >> j) & 1 for j in reversed(range(self.NUM_QUBITS))]
        #     dirs = B
        #     if self.NUM_QUBITS <= 10:
        #         for dir in seq:
        #             dirs = self.L @ dirs if dir == 0 else self.R @ dirs
        #     else:
        #         for d1, d2 in zip(seq[::2], seq[1::2]):
        #             if d1 == 0 and d2 == 0:
        #                 dirs = self.LL @ dirs
        #             elif d1 == 0 and d2 == 1:
        #                 dirs = self.RL @ dirs
        #             elif d1 == 1 and d2 == 0:
        #                 dirs = self.LR @ dirs
        #             elif d1 == 1 and d2 == 1:
        #                 dirs = self.RR @ dirs
            
        #         if self.NUM_QUBITS % 2 == 1:
        #             d = seq[-1]
        #             dirs = self.L @ dirs if d == 0 else self.R @ dirs

        #     result = A @ dirs
            # val = result.toarray()[0, 0]
            # val = result.item()
            # print("L: ", self.L)
            # print("R: ", self.R)
            # print("Leaf: ", self.leaf)
            # print(f"Path: {seq}, Value: {val}")
            # if abs(val) > 1e-10:
            #     self.sim_result[tuple(seq)] = val
            # yield val, seq

    def results(self):
        return self.sim_result

def simulation_all_path(L, R, leaf, NUM_QUBITS, TYPE):
    sim = SimulationAllPath(L, R, leaf, NUM_QUBITS, TYPE)
    sim.simulation()
    # for val, seq in sim:
    #     pass  # Iterate through all paths to compute results
    return sim.results()



In [13]:
def dd_sparse(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int, TYPE: str):
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS, TYPE)
    # print("Init Mapping: ", Mapping)
    total_start = time.time()

    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
    
    for op in operations:
        if time.time() - total_start > 60:
            raise TimeoutError("Timeout: 60s.")
        
        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
        
        A = cp.ones(MATRIX_SIZE, dtype=complex)
        A[Mapping[op.target]] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=complex)
        B[Mapping[op.target]] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=complex)
        C[Mapping[op.target]] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=complex)
        D[Mapping[op.target]] = op.d

        A = sparse.diags(A, offsets=0, format='csr')
        B = sparse.diags(B, offsets=0, format='csr')
        C = sparse.diags(C, offsets=0, format='csr')
        D = sparse.diags(D, offsets=0, format='csr')
    

        L_U = L @ A + R @ B
        R_U = L @ C + R @ D

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)
        
        elif isinstance(op, ControlledGate):
            MATRIX_SIZE_OLD = MATRIX_SIZE
            for control in reversed(op.control):
                keep = cp.zeros(MATRIX_SIZE_OLD, dtype=int)
                keep_complement = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                keep[Mapping[control]] = 1
                keep_complement[Mapping[control]] = 0
                D = sparse.diags(keep, offsets=0, format='csr')
                D_complement = sparse.diags(keep_complement, offsets=0, format='csr')

                if control < op.target:
                    R_masked = R @ D_complement
                    R_U_masked = R_U[:, :MATRIX_SIZE_OLD] @ D

                    L_U = sparse.bmat([[L, None],
                        [None, L_U]], format='csr')
                
                    # print("Mapping: ", Mapping[control])
                    
                    # print("D: ", D)
                    # print("D_complement: ", D_complement)
                    # print("R: ", R)
                    # print("R_U: ", R_U)
                    # print("R_U[]: ", R_U[:, :MATRIX_SIZE_OLD])
                    # print("R_masked: ", R_masked)
                    # print("R_U_masked: ", R_U_masked)

                    
                    # if TYPE == 'cupy':
                    #     print("R Masked Type: ", type(R_masked))
                    #     print("R U Masked Type: ", type(R_U_masked))
                    #     print("R U Type: ", type(R_U))
                    #     print("R Masked shape: ", R_masked.shape)
                    #     print("R U Masked shape: ", R_U_masked.shape)
                    #     print("R_U shape: ", R_U.shape)
                        # print("R_masked max row/col:", R_masked.nonzero())
                        # print("R_U_masked max row/col:", R_U_masked.nonzero())
                        # print("R_U max row/col:", R_U.nonzero())

                    R_U = sparse.bmat([[R_masked, None],
                        [R_U_masked, R_U]], format='csr')
                    
                    # print("L_U: ", L_U)
                    # print("R_U: ", R_U)
                else:
                    
                    L_U_masked = L_U @ D_complement
                    L_masked = L[:, :MATRIX_SIZE_OLD] @ D
                    L_U = sparse.bmat([[L_U_masked, None],
                        [L_masked, L]], format='csr')

                    R_U = sparse.bmat([[R_U, None],
                        [None, R]], format='csr')
                
                MATRIX_SIZE += MATRIX_SIZE_OLD

            L, R = L_U, R_U
            # print("start traverse")
            print("L: ", L)
            print("R: ", R)
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)

            traverse = sparse.csr_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)

            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            Mapping_U = [
                sorted(
                    [n + MATRIX_SIZE_OLD * i
                    for i in range(1, MATRIX_SIZE // MATRIX_SIZE_OLD + 1)
                    for n in mapping
                    if n + MATRIX_SIZE_OLD * i in traverse] +
                    [n for n in mapping if n in traverse]
                )
                for mapping in Mapping
            ]
            # print("Mapping_U: ", Mapping_U)

            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            Mapping = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U after dict: ", Mapping_U)
            # Mapping = copy.deepcopy(Mapping_U)
            # print("end mapping, time spent:", time.time() - mapping_middle)  

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csr_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.complex128)
            
            L = S @ L @ S.T
            R = S @ R @ S.T

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")

        print("MATRIX_SIZE: ", MATRIX_SIZE)
        print("L: ", L)
        print("R: ", R)
        print("Mapping: ", Mapping)
    # sim_result = compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2], TYPE)
    sim_result = simulation_all_path(L, R, Mapping[-2], NUM_QUBITS, TYPE)
    # sim_result = 0
    return sim_result

In [15]:
def dd_sparse_numpy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int):
    
    def monitor_cpu_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)  # percent of a core
            mem = process.memory_info().rss / 1024**2  # in MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)
            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

    import numpy as cp
    import scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_cpu_usage, args=(stop_event, result_dict))

        np_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
            # if time.time() - np_start > 60:
            #     raise TimeoutError("Simulation Timeout: 60s.")
        sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'numpy')

        np_end = time.time()

        stop_event.set()
        monitor_thread.join()

        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = None

        print(sim_result)

        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))
    except Exception as e:
        np_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in numpy + scipy:", e)
        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = str(e)
        traceback.print_exc()

    # print("Time spent: ", np_end - np_start)
    return result_dict


In [16]:
def dd_sparse_cupy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int):
    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    import cupy as cp
    import cupyx.scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))
    
        cp_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
        #     if time.time() - cp_start > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")
        sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'cupy')
        
        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))

        device_id = cp.cuda.runtime.getDevice()
        device_props = cp.cuda.runtime.getDeviceProperties(device_id)
        # print(f"Device {device_id} name:", device_props['name'].decode())
        print(sim_result)
        
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = None

    except Exception as e:
        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in cupy computation:", e)
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = str(e)
        traceback.print_exc()


    # print("Time spent: ", cp_end - cp_start)
    return result_dict


In [17]:
# cudaq.set_target("nvidia")

# kernel = build_kernel_from_operations(NUM_QUBITS, operations)
# result = cudaq.sample(kernel)
# print(result)


In [18]:
def cudaq_sv(path: str, NUM_QUBITS):
    # NUM_QUBITS = min(NUM_QUBITS, 16)
    NUM_QUBITS = 0
    # output_path = Path(path + 'circuit_cudaq.py')
    # !npx q-convert -i {input_path} -s qasm -o {output_path} -d cudaq

    method_style_gates = {"x", "y", "z", "h", "rx", "ry", "rz", "t", "s", "swap"}

    def parse_qasm_to_cudaq_python(qasm_file: str, output_py_file: str):
        with open(qasm_file, 'r') as f:
            lines = f.readlines()

        cudaq_lines = [
            "import cudaq\n",
            "@cudaq.kernel\n",
            "def circuit():\n"
        ]
        indent = "    "

        for line in lines:
            line = line.strip()
            if not line or line.startswith("OPENQASM") or line.startswith("include") or line.startswith("//"):
                continue

            if match := re.match(r"qreg\s+(\w+)\[(\d+)\];", line):
                regname, size = match.groups()
                cudaq_lines.append(f"{indent}{regname} = cudaq.qvector({size})\n")
                continue

            if match := re.match(r"([a-zA-Z_][a-zA-Z0-9_]*)\s+(.*);", line):
                gate_raw, arglist = match.groups()
                args = [arg.strip() for arg in arglist.split(",") if arg.strip()]

                ctrl_count = len(gate_raw) - len(gate_raw.lstrip("c"))
                base_gate = gate_raw[ctrl_count:]

                if ctrl_count > 0:
                    controls = args[:ctrl_count]
                    targets = args[ctrl_count:]

                    if base_gate in method_style_gates and len(targets) == 1:
                        ctrl_list = f"[{', '.join(controls)}]"
                        cudaq_lines.append(
                            f"{indent}{base_gate}.ctrl({ctrl_list}, {targets[0]})\n"
                        )
                    else:
                        ctrl_tuple = ", ".join(controls)
                        lambda_call = f"{base_gate}({', '.join(targets)})"
                        cudaq_lines.append(
                            f"{indent}cudaq.control(({ctrl_tuple}), lambda: {lambda_call})\n"
                        )
                else:
                    if base_gate in method_style_gates:
                        cudaq_lines.append(f"{indent}{base_gate}({', '.join(args)})\n")
                    else:
                        cudaq_lines.append(f"{indent}# TODO: Unrecognized gate: {base_gate}({', '.join(args)})\n")

        cudaq_lines.append(f"\ncounts = cudaq.sample(circuit, shots_count={2**NUM_QUBITS})\nprint(counts)")
        with open(output_py_file, 'w') as f:
            f.writelines(cudaq_lines)

    
    input_path = Path(path + 'circuit.qasm')
    output_path = Path(path + 'circuit_cudaq_new.py')
    parse_qasm_to_cudaq_python(input_path, output_path)

    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)
        
        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))

        cudaq_start = time.time()
        monitor_thread.start()

        import subprocess
        # %run -i {output_path}
        result = subprocess.run(
            ['python3', output_path],
            capture_output=True,
            text=True,
            check=True  # raises CalledProcessError if returncode != 0
        )
        print("STDOUT: ", result.stdout)

        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict["error"] = None
        # most_probable_result = counts.most_probable()
        # probability = counts.probability(most_probable_result)
        # print("Most probable result: " + most_probable_result)
        # print("Measured with probability " + str(probability), end='\n\n')
        # print("time spent: ", cudaq_end - cudaq_start)
    # except Exception as e:
    except subprocess.CalledProcessError as e:
        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in cudaq computation:", e)
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict["error"] = str(e.stderr)
        print("Error output:\n", e.stderr)
        traceback.print_exc()
    return result_dict


In [19]:
dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)

MATRIX_SIZE:  7
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (7, 7)>
  Coords	Values
  (2, 0)	(1+0j)
  (3, 1)	(1+0j)
  (4, 2)	(1+0j)
  (5, 3)	(1+0j)
  (6, 4)	(1+0j)
R:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (7, 7)>
  Coords	Values
  (1, 0)	(1+0j)
  (4, 1)	(1+0j)
  (4, 2)	(1+0j)
  (6, 3)	(1+0j)
  (6, 4)	(1+0j)
Mapping:  [[0], [1, 2], [3, 4], [5], [6]]
MATRIX_SIZE:  7
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (7, 7)>
  Coords	Values
  (2, 0)	(1+0j)
  (3, 1)	(1+0j)
  (4, 2)	(1+0j)
  (6, 3)	(1+0j)
  (6, 4)	(1+0j)
R:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (7, 7)>
  Coords	Values
  (1, 0)	(1+0j)
  (4, 1)	(1+0j)
  (4, 2)	(1+0j)
  (5, 3)	(1+0j)
  (6, 4)	(1+0j)
Mapping:  [[0], [1, 2], [3, 4], [5], [6]]
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 15 stored e

/home/daniel/.local/lib/python3.11/site-packages/scipy/sparse/_index.py:108: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


{(1, 1, 1): np.complex128(1+0j)}


{'avg_cpu': 0.0,
 'peak_mem': 96.44921875,
 'time_spent': 0.0291290283203125,
 'error': None}

In [20]:
import pandas as pd
import openpyxl

manual =  False

# benchmarks = {
#               'BV': 1,
#             #   'BVSym': 1, 
#             #   'GHZall': 2, 
#               'GHZzero': 2, 
#               'Grover': 2, 
#             #   'GroverSym': 2, 
#             #   'H2': 1, 
#             #   'HXH': 1, 
#               'MCToffoli': 3, 
#             #   'MOBV_reorder': 1, 
#             #   'MOGrover': 3, 
#             #   'MOGroverSym': 3, 
#               'OEGrover': 2
#               }
benchmarks = {
              'BV': 1,
            #   'BVSym': 1, 
            #   'GHZall': 2, 
            #   'GHZzero': 2, 
            #   'Grover': 2, 
            #   'GroverSym': 2, 
            #   'H2': 1, 
            #   'HXH': 1, 
            #   'MCToffoli': 3, 
            #   'MOBV_reorder': 1, 
            #   'MOGrover': 3, 
            #   'MOGroverSym': 3, 
            #   'OEGrover': 2
              }

benchmark_data = {}

for benchmark in benchmarks:

    if manual:
        break

    methods = ['NumPy + PySparse', 'CuPy + cuSparse', 'CudaQ (State Vector)']
    subfields = ['Time (s)', 'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error']

    columns = [('Qubits', '')]
    rows = []

    for method in methods:
        for sub in subfields:
            if method == 'NumPy + PySparse' and sub == 'GPU Memory Usage (MB)':
                continue
            columns.append((method, sub))
    
    multi_columns = pd.MultiIndex.from_tuples(columns)

    flag = (True, True, True)
    
    while (flag[0] or flag[1] or flag[2]) and benchmarks[benchmark] <= 100:

        if benchmark == 'Grover' and benchmarks[benchmark] == 22:
            break

        if benchmark == 'GroverSym' and benchmarks[benchmark] == 22:
            break

        if benchmark == 'MOGrover' and benchmarks[benchmark] == 12:
            break

        if benchmark == 'MOGroverSym' and benchmarks[benchmark] == 12:
            break
        

        i = str(benchmarks[benchmark])
        if int(i) < 10:
            i = '0' + i
        if benchmark in ['GHZall', 'GHZzero', 'H2'] and int(i) < 100:
            i = '0' + i

        path = '../ext/AutoQ/benchmarks/all/' + benchmark + f'/{i}/'
        input_path = Path(path + 'circuit.qasm')
        NUM_QUBITS, operations = parse_qasm_to_custom(input_path)

        INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1
        print(f"Qubit count: {NUM_QUBITS}")
        # print("Operations:", operations)

        if flag[0]:
            np_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
        if flag[1]:
            cp_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
        if flag[2]:
            cudaq_result = cudaq_sv(path, NUM_QUBITS)

        rows.append([NUM_QUBITS, 
                     np_result['time_spent'] if flag[0] else '--',
                     np_result['peak_mem'] if flag[0] else '--',
                     np_result['error'] if flag[0] else '--',
                     cp_result['time_spent'] if flag[1] else '--',
                     cp_result['peak_mem'] if flag[1] else '--',
                     cp_result['peak_gpu_mem'] if flag[1] else '--',
                     cp_result['error'] if flag[1] else '--',
                     cudaq_result['time_spent'] if flag[2] else '--',
                     cudaq_result['peak_mem'] if flag[2] else '--',
                     cudaq_result['peak_gpu_mem'] if flag[2] else '--',
                     cudaq_result['error'] if flag[2] else '--'
                     ])
                   

        if not np_result['error'] is None:
            flag = (False, flag[1], flag[2])
        
        if not cp_result['error'] is None:
            flag = (flag[0], False, flag[2])
        
        if not cudaq_result['error'] is None:
            flag = (flag[0], flag[1], False)

        benchmarks[benchmark] += 1

    benchmark_data[benchmark] = rows
        

FileNotFoundError: [Errno 2] No such file or directory: 'ext/AutoQ/benchmarks/all/BV/01/circuit.qasm'

In [21]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

excel_output_path = './test.xlsx'

excel_file = Path(excel_output_path)

def write_sheets(writer, data_dict):
    for benchmark, rows in data_dict.items():
        df = pd.DataFrame(rows, columns=multi_columns)
        df.to_excel(writer, sheet_name=benchmark, index=True)

if not excel_file.exists():
    # Create a new Excel file if it does not exist
    with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
        write_sheets(writer, benchmark_data)
else:
    try:
        # Load the workbook
        book = load_workbook(excel_output_path)
        for benchmark in benchmark_data.keys():
            if benchmark in book.sheetnames:
                # Delete the sheet if it exists
                std = book[benchmark]
                book.remove(std)

        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer.book = book  # attach modified workbook
            write_sheets(writer, benchmark_data)

    except Exception as e:
        print(f"Error opening existing Excel file: {e}")
        print("Recreating the file in write mode.")
        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
            write_sheets(writer, benchmark_data)


Error opening existing Excel file: File is not a zip file
Recreating the file in write mode.


IndexError: At least one sheet must be visible

In [ ]:
print(benchmark_data)

{'BV': [[2, 0.012251615524291992, 119.328125, None, 0.46481990814208984, 321.078125, 316.5625, None, 6.2491395473480225, 621.4609375, 480.2890625, None], [3, 0.009400606155395508, 0.0, None, 0.06683802604675293, 621.7109375, 471.42578125, None, 6.338107347488403, 621.7109375, 471.42578125, None], [4, 0.015499353408813477, 621.7109375, None, 0.10822367668151855, 621.7109375, 471.42578125, None, 6.466290712356567, 621.7109375, 471.42578125, None], [5, 0.01093745231628418, 0.0, None, 0.13013100624084473, 621.9609375, 471.42578125, None, 6.351664304733276, 621.9609375, 471.42578125, None], [6, 0.01703047752380371, 621.9609375, None, 0.18509888648986816, 621.9609375, 471.42578125, None, 6.418133974075317, 621.9609375, 471.42578125, None], [7, 0.020769119262695312, 0.0, None, 0.1814265251159668, 621.9609375, 471.42578125, None, 6.175281047821045, 621.9609375, 684.2890625, None], [8, 0.020506620407104492, 621.9609375, None, 0.2373371124267578, 621.9609375, 471.42578125, None, 6.25472259521484

In [14]:
import os
from pathlib import Path
import pandas as pd
from typing import List, Dict, Tuple
import pandas as pd
import openpyxl

def find_all_benchmark_files() -> Dict[str, List[Path]]:
    """
    Find all benchmark files in the specified directories:
    - .qasm files in MQTbench 
    - .real files in Revlib_renamed
    - .qasm files in AutoQ/benchmark/all/ (if exists)
    
    Returns:
        Dictionary with file types as keys and lists of Path objects as values
    """
    base_path = Path("../ext/DD_benchmark")
    
    files_found = {
        'mqt_qasm': [],
        'revlib_real': [],
        'autoq_qasm': []
    }
    
    # Find MQTbench .qasm files
    mqt_path = base_path / "MQTbench"
    if mqt_path.exists():
        for qasm_file in mqt_path.rglob("*.qasm"):
            files_found['mqt_qasm'].append(qasm_file)
    
    # Find Revlib .real files  
    revlib_path = base_path / "Revlib_renamed"
    if revlib_path.exists():
        for real_file in revlib_path.rglob("*.real"):
            files_found['revlib_real'].append(real_file)
    
    # Find AutoQ benchmark .qasm files
    autoq_path = Path("../ext/AutoQ/benchmarks/all")
    if autoq_path.exists():
        for qasm_file in autoq_path.rglob("*.qasm"):
            files_found['autoq_qasm'].append(qasm_file)
    else:
        print("AutoQ benchmark directory does not exist. Skipping AutoQ files search.")
    return files_found

def iterate_all_benchmark_files():
    """
    Iterate through all benchmark files and provide summary statistics
    """
    print("🔍 Searching for benchmark files...")
    
    files_found = find_all_benchmark_files()
    
    # Print summary
    total_files = sum(len(file_list) for file_list in files_found.values())
    print(f"\n📊 Summary of files found:")
    print(f"  MQTbench (.qasm): {len(files_found['mqt_qasm'])} files")
    print(f"  Revlib (.real):   {len(files_found['revlib_real'])} files") 
    print(f"  AutoQ (.qasm):    {len(files_found['autoq_qasm'])} files")
    print(f"  Total files:      {total_files} files")
    
    return files_found

# Run the iterator


In [18]:
benchmark_files = iterate_all_benchmark_files()
for file in benchmark_files['mqt_qasm'][:5] + benchmark_files['autoq_qasm'][:5]:
    print(file)  # Print first 5 MQTbench .qasm files

🔍 Searching for benchmark files...

📊 Summary of files found:
  MQTbench (.qasm): 419 files
  Revlib (.real):   196 files
  AutoQ (.qasm):    8156 files
  Total files:      8771 files
../ext/DD_benchmark/MQTbench/DJ/dj_indep_qiskit_38.qasm
../ext/DD_benchmark/MQTbench/DJ/dj_indep_qiskit_50.qasm
../ext/DD_benchmark/MQTbench/DJ/dj_indep_qiskit_42.qasm
../ext/DD_benchmark/MQTbench/DJ/dj_indep_qiskit_76.qasm
../ext/DD_benchmark/MQTbench/DJ/dj_indep_qiskit_30.qasm
../ext/AutoQ/benchmarks/all/BVBugSym/circuit.qasm
../ext/AutoQ/benchmarks/all/H2Sym/circuit.qasm
../ext/AutoQ/benchmarks/all/H2BugSym/circuit.qasm
../ext/AutoQ/benchmarks/all/GHZzero/501/circuit.qasm
../ext/AutoQ/benchmarks/all/GHZzero/084/circuit.qasm


In [ ]:

benchmark_files = iterate_all_benchmark_files()
benchmark_data = {}
for file in benchmark_files['mqt_qasm'] + benchmark_files['revlib_real'] + benchmark_files['autoq_qasm']:
    methods = ['NumPy + PySparse', 'CuPy + cuSparse', 'CudaQ (State Vector)']
    subfields = ['Time (s)', 'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error']

    columns = [('Qubits', '')]
    rows = []

    for method in methods:
        for sub in subfields:
            if method == 'NumPy + PySparse' and sub == 'GPU Memory Usage (MB)':
                continue
            columns.append((method, sub))
    
    multi_columns = pd.MultiIndex.from_tuples(columns)

    flag = (True, True, True)
    
    NUM_QUBITS, operations = parse_qasm_to_custom(file)

    INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1
    print(f"Qubit count: {NUM_QUBITS}")
        # print("Operations:", operations)

    if flag[0]:
        np_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
    if flag[1]:
        cp_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
    if flag[2]:
        cudaq_result = cudaq_sv(path, NUM_QUBITS)

    rows.append([NUM_QUBITS, 
                np_result['time_spent'] if flag[0] else '--',
                np_result['peak_mem'] if flag[0] else '--',
                np_result['error'] if flag[0] else '--',
                cp_result['time_spent'] if flag[1] else '--',
                cp_result['peak_mem'] if flag[1] else '--',
                cp_result['peak_gpu_mem'] if flag[1] else '--',
                cp_result['error'] if flag[1] else '--',
                cudaq_result['time_spent'] if flag[2] else '--',
                cudaq_result['peak_mem'] if flag[2] else '--',
                cudaq_result['peak_gpu_mem'] if flag[2] else '--',
                cudaq_result['error'] if flag[2] else '--'
                ])
                   

    if not np_result['error'] is None:
        flag = (False, flag[1], flag[2])
        
    if not cp_result['error'] is None:
        flag = (flag[0], False, flag[2])
        
    if not cudaq_result['error'] is None:
        flag = (flag[0], flag[1], False)


    benchmark_data[file] = rows